# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget
# %matplotlib inline

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.utils.plotting_tools as pt

import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys

# Cosmology

In [ ]:
# Create cosmology object
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time
ii_cosmology = 0

# Lines

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyA-1206"], ref_line_id = "HI-LyA") # "SiIII-1206", "SiII-1190", "SiII-1193" # "HI-LyA-1206", "HI-LyA-1193", "HI-LyA-1190"

# Skewers

In [ ]:
L_box = 110.7  # Mpc
z_box = 2.0

In [ ]:
N_grid_skewers = 6
N_ppsk = 4096

points_grid_skewers = aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y")
points = jnp.reshape(points_grid_skewers, (N_grid_skewers**2, N_ppsk, 3))

In [ ]:
skewer_bundle = SkewerBundle(spatial_points=points, cosmologies=cosmology, a0=1/(1+z_box), x0=L_box)

# Fields

In [ ]:
N_absorvers = 20
Delta = 1.2 * (L_box / N_ppsk)

n0_HI = 5e-10
T0_HI = 1.5*10**4

In [ ]:
n0_SiIII = n0_HI
T0_SiIII = T0_HI

n0_SiII = n0_HI
T0_SiII = T0_HI

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_HI, L_box=L_box, base_seed=0)),    # UNITS: [K]
        "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_HI, L_box=L_box, base_seed=0)), # [cm^-3]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    },
    # "SiIII": {
    #     "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiIII, L_box=L_box, base_seed=0)),    # UNITS: [K]
    #     "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiIII, L_box=L_box, base_seed=0)), # [cm^-3]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
    # "SiII": {
    #     "temperature": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-5, C0=T0_SiII, L_box=L_box, base_seed=0)),    # UNITS: [K]
    #     "number_density": ScalarField(toy_samplers.periodic_top_hat_absorber_sampler(N_absorvers=N_absorvers, Delta=Delta, C_low=1e-20, C0=n0_SiII, L_box=L_box, base_seed=0)), # [cm^-3]
    #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
    #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
    #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
    # },
})

In [ ]:
# T0_HI = 1.5*10**4
# T0_Si = 1.5*10**4
# n0_min = 1e-16
# n0_max = 1e-10
# def pk_template(k: jnp.ndarray, A, n, k0, k_cut):
#     # P(k) = A (k/k0)^n exp[-(k/k_cut)^2]
#     kk = jnp.maximum(k, 1e-12)
#     return A * (kk / k0) ** n * jnp.exp(-(kk / k_cut) ** 2)
# number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
#     key=jax.random.PRNGKey(0),
#     pk_of_k=lambda k: pk_template(k, A=5.0, n=-2.5, k0=2.0, k_cut=50.0),
#     n0_min=n0_min, n0_max=n0_max, L_box=L_box,
#     grid_shape=(256, 256, 256), contrast=1.2,
# )
# fields = Fields({
#     "HI": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_HI, T0_HI, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     "SiIII": {
#         "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#         "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#         "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#         "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#         "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     },
#     # "SiII": {
#     #     "temperature": ScalarField(toy_samplers.temperature_sampler(T0_Si, T0_Si, L_box)),    # UNITS: [K]
#     #     "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),        # UNITS: [km/s]
#     #     "number_density": ScalarField(number_density_sampler),                          # [cm^-3]
#     #     "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),         # UNITS: [km/s]
#     #     "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))        # UNITS: [GHz]
#     # },
# })

### Plot Fields

In [ ]:
sampled_fields = fields.sample(skewer_bundle.quadri_vectors[ii_cosmology], skewer_bundle.los_directions)

In [ ]:
x_array = points[..., 1]

key_field_plot = 'number_density'
plot_dict = {}
for ii, key in enumerate(sampled_fields.keys()):
    plot_dict[key] = jnp.log10(sampled_fields[key][key_field_plot])

fig, ax = pt.plot_3d_radar_cylinder(
    x_array=x_array, y_dict=plot_dict, cylinder_radius=20.0, skewer_index=0, field_logscale=False,
    x_axis_label=r"$r_{\parallel}\,[{\rm cMpc}]$",
    field_units=r"$\log_{10}\left(n \, \left[\mathrm{cm}^{-3}\right] \right)$",
    n_samples_plot=1, n_y_slices=5, n_x_ticks=5, x_axis_pad=0.2, factor_ticks_z=15.0,
    factor_x_label=1.3, show_profile_points=False,
)

fig, ax = pt.plot_3d_neon_ridge_cube(
    points=points, y_dict=plot_dict, normalization="global", field_logscale=False, normalize_scales=False,
    x_label=r"$x\,[{\rm cMpc}]$", y_label=r"$r_{\parallel}\,[{\rm cMpc}]$", z_label=r"$z\,[{\rm cMpc}]$",
    axis_label_fontsize=22, x_label_pad = 25.0, y_label_pad = 25.0, z_label_pad = 25.0, n_axis_ticks=(4, 4, 4),
    ridge_scale=1.0,
)

# Optical depth

In [ ]:
N_v = 10000

In [ ]:
a_dagger = 1/(1+z_box)
a_em = skewer_bundle.scale_factors[0, 0, 0]
v_max = phys.c_kms * jnp.log(a_dagger / a_em)
velocities_kms = jnp.linspace(0, v_max, N_v)
dv = np.diff(velocities_kms)[0]

In [ ]:
tau = od.OpticalDepthCalculator(
    skewer_bundle=skewer_bundle, velocities_kms=velocities_kms, line_bundle=line_bundle, fields=fields, a_dagger=a_dagger, debug_memory=True,
)
tau.compute()
tau.wrap_to_ref_line()
optical_depth = tau.optical_depth

In [ ]:
# # THIS IS JUST A TOY MODEL FOR CHECKING WHAT HAPPENS WHEN THE OPTICAL DEPTH HAS A SINGLE PEAK, IT IS NOT SUPPOSED TO BE PHYSICALLY REALISTIC
# N_sk = skewer_bundle.N_sk
# peak_indices = jax.random.randint(jax.random.PRNGKey(0), shape=(N_sk,), minval=0, maxval=N_v)
# toy_optical_depth = jnp.zeros((N_sk, N_v), dtype=jnp.float32)
# toy_optical_depth = 100 * toy_optical_depth.at[jnp.arange(N_sk), peak_indices].set(1.0)
# toy_optical_depth = jnp.tile(toy_optical_depth[jnp.newaxis, :], (2, 1, 1))

# # Periodically wrap by hand the second entry by `shift_pixels` pixels
# shift_pixels = int(line_bundle.delta_v[1] / dv) 
# toy_optical_depth = toy_optical_depth.at[1].set(jnp.roll(toy_optical_depth[1], shift=shift_pixels, axis=-1))

# tau = sjx.optical_depth.optical_depth.OpticalDepthCalculator(optical_depth=toy_optical_depth, velocities_kms=velocities_kms, line_bundle=line_bundle)
# # tau.wrap_to_ref_line()
# optical_depth = tau.optical_depth

### Plot optical depth

In [ ]:
x_array = jnp.tile(velocities_kms[None], (tau.N_sk, 1))

plot_dict = {}
for ii, line in enumerate(line_bundle.lines):
    plot_dict[line.name] = jnp.log10(optical_depth[ii])
    
fig, ax = pt.plot_3d_radar_cylinder(
    x_array=x_array, y_dict=plot_dict, cylinder_radius=2000.0, skewer_index=0, field_logscale=False,
    factor_ticks_x=3.0, factor_ticks_z=1200.0, factor_x_label=1.3, x_axis_label=r"$v\,[{\rm km / s}]$",
    field_units=r"$\log_{10}\tau$", n_samples_plot=1, n_y_slices=5, n_x_ticks=5, x_axis_pad=0.2,
    show_profile_points=False,
)

x_trans = points[:tau.N_sk, 0, 0]
z_trans = points[:tau.N_sk, 0, 2]

x_cube = jnp.tile(x_trans[:, None], (1, N_v))
y_cube = jnp.tile(velocities_kms[None, :], (tau.N_sk, 1))
z_cube = jnp.tile(z_trans[:, None], (1, N_v))

points_cube_velocities = jnp.stack([x_cube, y_cube, z_cube], axis=-1)

fig, ax = pt.plot_3d_neon_ridge_cube(
    points=points_cube_velocities, y_dict=plot_dict, normalization="global", field_logscale=False, normalize_scales=True,
    x_label=r"$x\,[{\rm cMpc}]$", y_label=r"$v\,[{\rm km\,s^{-1}}]$", z_label=r"$z\,[{\rm cMpc}]$", axis_label_fontsize=22,
    ridge_scale=1.0, x_label_pad = 25.0, y_label_pad = 25.0, z_label_pad = 25.0, n_axis_ticks=(4, 4, 4),
)

fig, axes, idx = pt.plot_samples_neon(
    yy=np.exp(-optical_depth), line_bundle=line_bundle, x_axis_array=velocities_kms,
    x_label=r"$v \, \left[\mathrm{km} \, \mathrm{s}^{-1}\right]$", N_plot_sample=5, seed=137,
    line_lw=3.5, line_glow_lw=6.0, line_alpha=0.95,line_glow_alpha=0.1, line_legend_lw=2.4,
    total_color="gold", total_lw=1.5, total_glow_lw=2.0, total=None
)

# Overflux

In [ ]:
overflux = sjx.optical_depth.overflux.DeltaFieldBuilder(optical_depth).build()

# 1D summary stats

In [ ]:
delta_v_list = list(np.array(line_bundle.delta_v))
velocity_domain = np.max(tau.velocities_kms) - np.min(tau.velocities_kms)
stats = sjx.summary_stats.P1D_xi1D.OverfluxStatistics1D(overflux, dv)
ordered_pairs = stats.ordered_pairs

# $P 1D$

In [ ]:
stats.compute_p1d()

Pk_total_from_terms = stats.Pk_total_from_terms

Pk_total_per_sample = stats.Pk_total_per_sample
Pk_total = stats.Pk_total

Pk_pair_per_sample = stats.Pk_pair_per_sample
Pk_pair = stats.Pk_pair

k = stats.k

Angular wavenumber: $k_m = 2\pi f_m = \frac{2\pi m}{N_v dv}$

In [ ]:
fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_by_order(
    ordered_pairs=ordered_pairs, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=velocity_domain, Npix=tau.N_v,
    tolerance_threshold=1e-6, use_log_x=True, # xlim=(2.0, 600),
    residual_mode="sum", show_harmonics=False, use_symlog_main=False, use_symlog_residual=False,
    symlog_linscale=1.0, symlog_label_minor=False, symlog_sci_cutoff=1e-2, legend_ncol_terms=2,
    xlabel=r"$k\,[{\rm s\,/km}]$", ylabel=r"$P_{\rm 1D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_p1d_total_recovery(
    Pk_total=Pk_total, Pk_total_from_terms=Pk_total_from_terms, Pk_pair=Pk_pair, k=k, delta_v_list=delta_v_list, L_box=velocity_domain, Npix=tau.N_v,
    ordered_pairs=ordered_pairs, show_harmonics=False, show_order_sums=True, show_order_sum_residuals=True,
    use_symlog_main=True, use_symlog_residual=False, xlabel=r"$k\,[{\rm s\,/km}]$"
)

In [ ]:
# ===============================================================
# Build order sums from Pk_pair

ordered_pairs = list(Pk_pair.keys()) if ordered_pairs is None else ordered_pairs
order_sum_dict = {}
for pair in ordered_pairs:
    A, B = pair
    order = len(A) + len(B)
    if order not in order_sum_dict:
        order_sum_dict[order] = np.zeros_like(Pk_pair[pair], dtype=float)
    order_sum_dict[order] += Pk_pair[pair].real
orders = sorted(order_sum_dict.keys())

# ===============================================================
# Plot P1D

cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(12, 9))
for i, order in enumerate(orders):
    y_order = order_sum_dict[order][1:] * k[1:] / np.pi
    ax.plot(k[1:], y_order, lw=2.0, color=cmap(i % 10), alpha=0.9, label=rf"$\mathcal{{O}}({order})$")

# Total
ax.plot(k[1:], Pk_total.real[1:] * k[1:] / np.pi, lw=3.0, color="k", label=r"Tot.")

# Total recovered from terms
ax.plot(k[1:], Pk_total_from_terms.real[1:] * k[1:] / np.pi, lw=2.0, color="red", ls="--", label=r"Tot. Sum")

ax.set_yscale("log")

ax.set_xlabel(r"$k\,[{\rm s\,/km}]$", fontsize=22)
ax.set_ylabel(r"$k P_{\rm 1D} / \pi$", fontsize=22)

# ax.set_xlim([0.0, 0.045])
# ax.set_ylim([0.007, 0.6])

ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
Jonas2026_x = np.asarray([0.0012103424520726075, 0.001710426016737391, 0.0022405113342219723, 0.0026967336406882466, 0.0032317105483064905, 0.003716793236561375, 0.004218507331270713, 0.004720058372975595, 0.005196173145985431, 0.005711909799077935, 0.006196503328319453, 0.006692021408859485, 0.007234824860691589, 0.007707515520607861, 0.00821346899343304, 0.008739967144819602, 0.009207603161597756, 0.009733122994957585, 0.010237935096751576, 0.010697581516311415, 0.011248863724375202, 0.011746175387964242, 0.012217072464831506, 0.01268682817066758, 0.01319946681667543, 0.013692702155153086, 0.01418707886466193, 0.01473575222465443, 0.015217410799815749, 0.01573249524089043, 0.01624921021200967, 0.01672907520412198, 0.01725997578662884, 0.01769777310359165, 0.018203237417403463, 0.01873022472780339, 0.019218568476147384, 0.019730717963141867, 0.020171124128175965, 0.020730558986462522, 0.021214663356690675, 0.021757303755518324, 0.022217928493104895, 0.02264920368988949, 0.023259348032561685, 0.023726331837322017, 0.024201305239300665, 0.024734325510865445, 0.025247616368891116])
Jonas2026_y = np.asarray([0.008702200282951175, 0.010605036156919673, 0.015611253374371708, 0.01789981585681677, 0.020231052528309375, 0.01894901896933219, 0.019569830478807133, 0.02324977718276436, 0.026511191625119498, 0.029135409525467774, 0.02720121897976352, 0.026378752285570163, 0.02845338682745897, 0.032664334384986116, 0.034334887584939515, 0.03291509367959117, 0.03325243646690979, 0.03226665029085164, 0.03275214068403131, 0.0360022415822107, 0.03663308200482874, 0.034491706603143844, 0.0338068641820707, 0.03275361843520383, 0.03600305376965098, 0.03683943699143153, 0.03718343175764788, 0.03466253256400984, 0.035570337794891445, 0.036471463316681864, 0.03717420582883128, 0.03783667637903175, 0.038286596603684, 0.03485464210089554, 0.036262260926746316, 0.03711303718723162, 0.03572393353222253, 0.03583774633183837, 0.03456882478160373, 0.03329146690260813, 0.03444272099734688, 0.03452985467990668, 0.03375885119909169, 0.03508816258046259, 0.03445748710264468, 0.03294778143947462, 0.03183069446317984, 0.0358676721137433, 0.03263340016247513])

In [ ]:
# ===============================================================
# Plot P1D

cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(12, 9))

# Total
ax.plot(k[1:], Pk_total.real[1:] * k[1:] / np.pi, lw=3.0, color="k", label=r"Toy Model")

ax.plot(Jonas2026_x, Jonas2026_y, lw=2.0, color="red", ls="-", label=r"Chaves-Montero et al. (2026) -- $z=2.2$")

ax.set_yscale("log")

ax.set_xlabel(r"$k\,[{\rm s\,/km}]$", fontsize=22)
ax.set_ylabel(r"$k P_{\rm 1D} / \pi$", fontsize=22)

ax.set_xlim([0.0, 0.03])
# ax.set_ylim([0.007, 0.06])

ax.tick_params(axis="both", labelsize=20)
ax.legend(fontsize=20, ncol=1)

plt.tight_layout()
plt.show()

# $\xi 1D$

In [ ]:
stats.compute_xi1d()

xi_total_from_terms = stats.xi_total_from_terms

xi_total_per_sample = stats.xi_total_per_sample
xi_total = stats.xi_total

xi_pair_per_sample = stats.xi_pair_per_sample
xi_pair = stats.xi_pair

lags = stats.lags

In [ ]:
fig, axes, axes_res, xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_by_order(
    ordered_pairs=ordered_pairs, xi_pair=xi_pair, lags=lags, delta_v_list=delta_v_list, L_box=velocity_domain,
    tolerance_threshold=1e-6, residual_mode="sum", show_delta_v_diff_lines=True, use_symlog_main=False,
    use_symlog_residual=False, symlog_linscale=1.0, symlog_base=10, symlog_label_minor=False, symlog_sci_cutoff=1e-2,
    show_order_badge=True, legend_ncol_terms=2, xlabel=r"$v\,\left[{\rm km / s}\right]$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
    xlim=(-0.0, 3000), # xlim=(-0.0, v_max/2)
)

fig, (ax, ax_res), xi_order_sum_dict, xi_order_sum_plot_dict = sjx.utils.plotting_tools_P1D_xi1D.plot_xi_total_recovery(
    xi_total=xi_total, xi_total_from_terms=xi_total_from_terms, xi_pair=xi_pair, lags=lags, delta_v_list=delta_v_list, L_box=velocity_domain, ordered_pairs=ordered_pairs,
    show_order_sums=True, show_order_sum_residuals=True, show_delta_v_diff_lines=True, annotate_half_line=False,
    half_line_text=r"$L_\mathrm{box}/2$", use_symlog_main=False, use_symlog_residual=True, main_symlog_linthresh=None,
    symlog_linscale=1.0, symlog_base=10, symlog_sci_cutoff=1e-2, xlabel=r"$v\,\left[{\rm km / s}\right]$", ylabel=r"$\xi_{\rm 1D}$", residual_ylabel=r"$\Delta$",
    xlim=(-0.0, 3000), # xlim=(-0.0, v_max/2)
)

# 1D Models

In [ ]:
from skewerjax.models import models

In [ ]:
# ===============================================================
# Input data
# ===============================================================

p_lya = np.asarray(Pk_pair[("1", "1")]).real
p_tot = np.asarray(Pk_total).real

# The sign of Delta v does not matter inside cos(k Delta v).
delta_v_siiii = float(np.abs(np.asarray(line_bundle.delta_v[1])))

k_fit_range = (0.0, 0.1)


# ===============================================================
# Optimizer tolerance parameters
# ===============================================================

mcdonald_optimizer_options = {
    "xatol": 1.0e-14,
    "maxiter": 5000,
}

ma_global_optimizer_options = {
    "tol": 1.0e-10,
    "atol": 1.0e-12,
    "maxiter": 1000,
    "popsize": 25,
    "polish": False,
    "seed": 12345,
    "workers": 1,
    "updating": "immediate",
}

ma_polish_optimizer_options = {
    "maxiter": 50000,
    "ftol": 1.0e-15,
    "gtol": 1.0e-13,
    "maxls": 100,
}


# ===============================================================
# Fit McDonald model
# ===============================================================

fit_mcdonald = models.fit_mcdonald_a(
    k=k,
    p_tot=p_tot,
    p_lya=p_lya,
    delta_v=delta_v_siiii,
    a_bounds=(-10.2, 10.2),
    k_range=k_fit_range,
    fit_mode="ratio",
    optimizer_options=mcdonald_optimizer_options,
)

a_mcd_best = fit_mcdonald["a"]

Pk_mcdonald_best = models.mcdonald_p1d(
    k=k,
    p_lya=p_lya,
    a=a_mcd_best,
    delta_v=delta_v_siiii,
)

ratio_mcdonald_best = models.mcdonald_factor(
    k=k,
    a=a_mcd_best,
    delta_v=delta_v_siiii,
)


# ===============================================================
# Fit Ma et al. profiled model
# ===============================================================

fit_ma = models.fit_ma_model_profiled(
    k=k,
    p_tot=p_tot,
    p_lya=p_lya,
    delta_v=delta_v_siiii,
    k_range=k_fit_range,
    fit_mode="ratio",

    # lambda_auto = 1 / k_auto
    # lambda_cross_abs = 1 / abs(k_cross)
    #
    # The lower bound 0.0 includes the McDonald-like constant limit.
    lambda_bounds={
        "lambda_auto": (0.0, 500.0),
        "lambda_cross_abs": (0.0, 500.0),
    },

    amplitude_bounds={
        "a_auto": (0.0, 100.0),
        "a_cross": (-100.0, 100.0),
    },

    enforce_positive_model=False,
    linear_lsq_tol=1.0e-14,
    run_global=True,
    global_optimizer_options=ma_global_optimizer_options,
    polish_optimizer_options=ma_polish_optimizer_options,
)

a_auto_best = fit_ma["a_auto"]
a_cross_best = fit_ma["a_cross"]
k_auto_best = fit_ma["k_auto"]
k_cross_best = fit_ma["k_cross"]

Pk_ma_auto, Pk_ma_cross, Pk_ma_best = models.ma_components(
    k=k,
    p_lya=p_lya,
    a_auto=a_auto_best,
    a_cross=a_cross_best,
    k_auto=k_auto_best,
    k_cross=k_cross_best,
    delta_v=delta_v_siiii,
)

ratio_ma_best = models.ma_factor(
    k=k,
    a_auto=a_auto_best,
    a_cross=a_cross_best,
    k_auto=k_auto_best,
    k_cross=k_cross_best,
    delta_v=delta_v_siiii,
)

ratio_ma_auto = Pk_ma_auto / p_lya
ratio_ma_cross = Pk_ma_cross / p_lya


# ===============================================================
# Print fit summary
# ===============================================================

print("\nMcDonald fit")
print("============")
print(f"a       = {fit_mcdonald['a']:.8e}")
print(f"chi2    = {fit_mcdonald['chi2']:.8e}")
print(f"N_fit   = {fit_mcdonald['n_fit']}")
print(f"dof     = {fit_mcdonald['dof']}")
print(f"mode    = {fit_mcdonald['fit_mode']}")
print(f"k range = {fit_mcdonald['k_range']}")
print(f"Delta v = {fit_mcdonald['delta_v']:.6f} km/s")
print(f"success = {fit_mcdonald['result'].success}")
print(f"message = {fit_mcdonald['result'].message}")
print(f"xatol   = {mcdonald_optimizer_options['xatol']:.3e}")
print(f"maxiter = {mcdonald_optimizer_options['maxiter']}")

print("\nMa et al. profiled fit")
print("======================")
print(f"a_auto          = {fit_ma['a_auto']:.8e}")
print(f"a_cross         = {fit_ma['a_cross']:.8e}")
print(f"k_auto          = {fit_ma['k_auto']:.8e} s/km")
print(f"k_cross         = {fit_ma['k_cross']:.8e} s/km")
print(f"lambda_auto     = {fit_ma['lambda_auto']:.8e} km/s")
print(f"lambda_crossabs = {fit_ma['lambda_cross_abs']:.8e} km/s")
print(f"chi2            = {fit_ma['chi2']:.8e}")
print(f"N_fit           = {fit_ma['n_fit']}")
print(f"dof             = {fit_ma['dof']}")
print(f"mode            = {fit_ma['fit_mode']}")
print(f"k range         = {fit_ma['k_range']}")
print(f"Delta v         = {fit_ma['delta_v']:.6f} km/s")
print(f"best candidate  = {fit_ma['best_candidate']['label']}")
print(f"linear success  = {fit_ma['lsq_result'].success}")
print(f"linear message  = {fit_ma['lsq_result'].message}")

print("\nModel comparison")
print("================")
print(f"chi2 McDonald = {fit_mcdonald['chi2']:.8e}")
print(f"chi2 Ma       = {fit_ma['chi2']:.8e}")
print(f"Delta chi2    = {fit_ma['chi2'] - fit_mcdonald['chi2']:.8e}")

if fit_ma["chi2"] > fit_mcdonald["chi2"]:
    print("\nWARNING")
    print("=======")
    print("The Ma profiled fit is still worse than McDonald.")
    print("This should not normally happen in ratio mode if:")
    print("  1. lambda_auto = lambda_cross_abs = 0 is allowed,")
    print("  2. a_auto bounds include a_McDonald^2,")
    print("  3. a_cross bounds include 2 a_McDonald,")
    print("  4. enforce_positive_model is False or not excluding the McDonald limit.")

In [ ]:
cmap = plt.get_cmap("tab10")

mask = (
    np.isfinite(k) & np.isfinite(p_tot) & np.isfinite(p_lya)
    & np.isfinite(Pk_mcdonald_best) & np.isfinite(Pk_ma_best)
    & (p_tot != 0) & (p_lya > 0)
    & (k > k_fit_range[0]) & (k < k_fit_range[1])
)

ratio_data = p_tot / p_lya
mask_r = mask & np.isfinite(ratio_data) & np.isfinite(ratio_mcdonald_best) & np.isfinite(ratio_ma_best) & (ratio_data != 0)

res_lya = (p_lya - p_tot) / p_tot
res_mcd = (Pk_mcdonald_best - p_tot) / p_tot
res_ma = (Pk_ma_best - p_tot) / p_tot
res_ratio_mcd = (ratio_mcdonald_best - ratio_data) / ratio_data
res_ratio_ma = (ratio_ma_best - ratio_data) / ratio_data

mcd_label = rf"McDonald fit: $a={a_mcd_best:.3e}$"
ma_label = (
    r"Ma fit: "
    rf"$a_{{\rm auto}}={a_auto_best:.2e}$, "
    rf"$a_{{\rm cross}}={a_cross_best:.2e}$, "
    rf"$k_{{\rm auto}}={k_auto_best:.2e}$, "
    rf"$k_{{\rm cross}}={k_cross_best:.2e}$"
)

# ===============================================================
# Plot 1: P1D with residual panel
# ===============================================================

fig, (ax, axr) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"height_ratios": [3.5, 1.0], "hspace": 0.05})

ax.plot(k[mask], p_tot[mask] * k[mask] / np.pi, lw=1.5, color="k", label=r"Total")
ax.plot(k[mask], p_lya[mask] * k[mask] / np.pi, lw=1.5, ls="-", color=cmap(0), label=r"Ly$\alpha$ only")
ax.plot(k[mask], Pk_mcdonald_best[mask] * k[mask] / np.pi, lw=2.5, ls="--", color=cmap(1), label=mcd_label)
ax.plot(k[mask], Pk_ma_best[mask] * k[mask] / np.pi, lw=3.0, ls=":", color=cmap(2), label=ma_label)
ax.set_yscale("log")
ax.set_ylabel(r"$k P_{\rm 1D} / \pi$", fontsize=22)
ax.tick_params(axis="both", labelsize=18)
ax.legend(fontsize=13)

axr.plot(k[mask], res_lya[mask], lw=1.5, ls="-", color=cmap(0), label=r"Ly$\alpha$")
axr.plot(k[mask], res_mcd[mask], lw=2.5, ls="--", color=cmap(1), label=r"McDonald")
axr.plot(k[mask], res_ma[mask], lw=3.0, ls=":", color=cmap(2), label=r"Ma")
axr.axhline(0.0, lw=1.2, color="0.5", alpha=0.8)
axr.set_xlabel(r"$k\,[{\rm s\,km^{-1}}]$", fontsize=22)
axr.set_ylabel(r"$\Delta P/P_{\rm tot}$", fontsize=18)
axr.set_xlim(k_fit_range)
axr.tick_params(axis="both", labelsize=16)

plt.tight_layout()
plt.show()

# ===============================================================
# Plot 2: P_tot / P_Lya with residual panel
# ===============================================================

fig, (ax, axr) = plt.subplots(2, 1, figsize=(12, 9), sharex=True, gridspec_kw={"height_ratios": [3.5, 1.0], "hspace": 0.05})

ax.plot(k[mask_r], ratio_data[mask_r], lw=1.5, color="k", label=r"Total: $P_{\rm tot}/P_{\rm Ly\alpha}$")
ax.plot(k[mask_r], ratio_mcdonald_best[mask_r], lw=2.5, ls="--", color=cmap(1), label=mcd_label)
ax.plot(k[mask_r], ratio_ma_best[mask_r], lw=3.0, ls=":", color=cmap(2), label=ma_label)
ax.axhline(1.0, lw=1.2, color="0.5", alpha=0.8)
ax.set_ylabel(r"$P_{\rm tot}(k)/P_{\rm Ly\alpha}(k)$", fontsize=22)
ax.tick_params(axis="both", labelsize=18)
ax.legend(fontsize=13)

axr.plot(k[mask_r], res_ratio_mcd[mask_r], lw=2.5, ls="--", color=cmap(1), label=r"McDonald")
axr.plot(k[mask_r], res_ratio_ma[mask_r], lw=3.0, ls=":", color=cmap(2), label=r"Ma")
axr.axhline(0.0, lw=1.2, color="0.5", alpha=0.8)
axr.set_xlabel(r"$k\,[{\rm s\,km^{-1}}]$", fontsize=22)
axr.set_ylabel(r"$\Delta R/R_{\rm tot}$", fontsize=18)
axr.set_xlim(k_fit_range)
axr.tick_params(axis="both", labelsize=16)

plt.tight_layout()
plt.show()


# 3D summary stats

In [ ]:
dx_perp = L_box / N_grid_skewers                       # transverse spacing [Mpc]

# ── Instantiate ─────────────────────────────────────────────────────────────
stats3d = sjx.summary_stats.P3D_xi3D.OverfluxStatistics3D(
    overflux, dv=dv, dx_perp=dx_perp, N_xy=N_grid_skewers, n_k_bins=20, n_mu_bins=16,
)
ordered_pairs = stats3d.ordered_pairs

# $P3D$

In [ ]:
stats3d.compute_p3d()
Pk3d_total             = stats3d.Pk3d_total           # (n_k_bins, n_mu_bins)
Pk3d_pair              = stats3d.Pk3d_pair            # dict (a,b) → same
Pk3d_total_from_terms  = stats3d.Pk3d_total_from_terms
k_bin_centers          = stats3d.k_bin_centers
mu_bin_centers         = stats3d.mu_bin_centers
Pk3d_counts            = stats3d.Pk3d_counts
k_iso, Pk_iso          = stats3d.isotropic_p3d()

In [ ]:
%autoreload

In [ ]:
fig, axes, axes_res, order_sum_dict = sjx.utils.plotting_tools_P3D_xi3D.plot_p3d_by_order(
    ordered_pairs=ordered_pairs, Pk3d_pair=Pk3d_pair, k_bin_centers=k_bin_centers, mu_bin_centers=mu_bin_centers,
    mu_wedges=None,           # default: 4 evenly spaced μ bins
    tolerance_threshold=1e-6, residual_mode="sum", use_log_x=True, use_symlog_main=False, use_symlog_residual=False,
    xlabel=r"$k\,[\mathrm{Mpc}^{-1}]$", ylabel=r"$P_{\rm 3D}$", residual_ylabel=r"$\Delta$",
)

fig, (ax, ax_res), order_sum_dict = sjx.utils.plotting_tools_P3D_xi3D.plot_p3d_total_recovery(
    Pk3d_total=Pk3d_total, Pk3d_total_from_terms=Pk3d_total_from_terms, Pk3d_pair=Pk3d_pair,
    k_bin_centers=k_bin_centers, mu_bin_centers=mu_bin_centers, ordered_pairs=ordered_pairs,
    show_order_sums=True, show_order_sum_residuals=True, use_log_x=True, use_symlog_main=True, use_symlog_residual=False,
    xlabel=r"$k\,[\mathrm{Mpc}^{-1}]$",
)

# $\xi3D$

In [ ]:
stats3d.compute_xi3d()
xi3d_total             = stats3d.xi3d_total           # (N_grid_skewers, N_grid_skewers, Npix)
xi3d_pair              = stats3d.xi3d_pair            # dict (a,b) → same
xi3d_total_from_terms  = stats3d.xi3d_total_from_terms
r3d_grid               = stats3d.r3d_grid
r_perp, pi, xi_rppi    = stats3d.xi3d_rppi()         # projected (r_perp, π)

In [ ]:
xi3d_pair_rppi = {
    pair: stats3d.xi3d_rppi.__func__(stats3d, field=pair[0])[2]   # or compute manually
    for pair in ordered_pairs
}
r_perp, pi, xi3d_total_rppi         = stats3d.xi3d_rppi(field="total")
r_perp, pi, xi3d_from_terms_rppi    = stats3d.xi3d_rppi(field="total")  # swap for decomp


In [ ]:
fig, axes_grid, order_sum_dict = sjx.utils.plotting_tools_P3D_xi3D.plot_xi3d_rppi_by_order(
    ordered_pairs=ordered_pairs, xi3d_pair=xi3d_pair_rppi,    # dict (a,b) -> (n_rperp, n_pi)
    r_perp=r_perp, pi=pi, r_perp_1d_indices=None,             # default: [0, n//4, n//2]
    use_symlog_1d=True, xlabel_map=r"$r_\perp\,[\mathrm{Mpc}]$", ylabel_map=r"$\pi\,[\mathrm{Mpc}]$",
    xlabel_1d=r"$\pi\,[\mathrm{Mpc}]$", ylabel_1d=r"$\xi_{\rm 3D}$", unit_label="Mpc",
)

fig, (ax_map_tot, ax_map_dec), axes_1d, axes_res, order_sum_dict = (
    sjx.utils.plotting_tools_P3D_xi3D.plot_xi3d_rppi_total_recovery(
        xi3d_total_rppi=xi3d_total_rppi, xi3d_total_from_terms_rppi=xi3d_from_terms_rppi, xi3d_pair_rppi=xi3d_pair_rppi,
        r_perp=r_perp, pi=pi, ordered_pairs=ordered_pairs, show_order_sums=True, show_order_sum_residuals=True, use_symlog_1d=True,
        xlabel_map=r"$r_\perp\,[\mathrm{Mpc}]$", ylabel_map=r"$\pi\,[\mathrm{Mpc}]$", xlabel_1d=r"$\pi\,[\mathrm{Mpc}]$", ylabel_1d=r"$\xi_{\rm 3D}$",
    )
)